# 01 · Data Setup
This notebook covers the very first step of every ML project: **getting to know your data**.

We'll:
1. Load the raw dataset and inspect its structure
2. Check for data quality issues (nulls, wrong dtypes)
3. Understand the class imbalance -- the core challenge of fraud detection
4. Perform a **stratified train/test split** so the fraud rate is identical in both halves
5. Save the splits to disk for all subsequent notebooks

> **Dataset**: [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
> 284,807 European cardholder transactions from September 2013.
> Features V1-V28 are PCA-transformed (anonymised for privacy).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data")
RANDOM_STATE = 42


## 1 · Load the raw data

We start by loading `creditcard.csv` and immediately inspecting its dimensions
and memory footprint -- good habits before touching a new dataset.


In [2]:
df = pd.read_csv(DATA_DIR / "creditcard.csv")

print(f"Shape : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
df.head()


Shape : 284,807 rows  x  31 columns
Memory: 67.4 MB


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 2 · Data types

All 31 columns should be numeric. `Class` is the binary target (0 = legit, 1 = fraud).


In [3]:
print(df.dtypes.to_string())


Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64


## 3 · Null check

One of the first things to verify: are there any missing values?
Missing fraud labels or feature values would silently corrupt training.


In [4]:
nulls = df.isnull().sum()
if nulls.any():
    print(nulls[nulls > 0])
else:
    print("No nulls found -- dataset is complete.")


No nulls found -- dataset is complete.


## 4 · Class distribution & fraud rate

This dataset is **severely imbalanced**.  Understanding the ratio now explains
every metric choice we make later (why we use AUC-ROC and PR-AUC instead of accuracy).


In [5]:
counts     = df["Class"].value_counts().sort_index()
total      = len(df)
fraud_n    = int(counts[1])
fraud_rate = fraud_n / total * 100

print("Class distribution")
print(f"  Legit (0) : {int(counts[0]):>7,}  ({100 - fraud_rate:.4f}%)")
print(f"  Fraud (1) : {fraud_n:>7,}  ({fraud_rate:.4f}%)")
print(f"  Total     : {total:>7,}")
print(f"\nFraud rate : {fraud_rate:.4f}%")
print(f"Imbalance  : {int(counts[0]) // fraud_n}:1  (legit:fraud)")
print()
print("Key insight: a naive model that always predicts 'legit' achieves")
print(f"{100 - fraud_rate:.2f}% accuracy -- accuracy is useless here.")
print("We will use AUC-ROC, PR-AUC, Precision, Recall, and F1 instead.")


Class distribution
  Legit (0) : 284,315  (99.8273%)
  Fraud (1) :     492  (0.1727%)
  Total     : 284,807

Fraud rate : 0.1727%
Imbalance  : 577:1  (legit:fraud)

Key insight: a naive model that always predicts 'legit' achieves
99.83% accuracy -- accuracy is useless here.
We will use AUC-ROC, PR-AUC, Precision, Recall, and F1 instead.


## 5 · Basic statistics for Amount and Time

`Amount` and `Time` are the only non-PCA columns.  Understanding their distributions
informs the feature engineering choices in the next notebook.


In [6]:
df[["Time", "Amount"]].describe().round(2)


,Time,Amount
count,284807.00,284807.00
mean,94813.86,88.35
std,47488.15,250.12
min,0.00,0.00
25%,54201.50,5.60
50%,84692.00,22.00
75%,139320.50,77.16
max,172792.00,25691.16


In [7]:
fraud = df[df["Class"] == 1]
legit = df[df["Class"] == 0]

print(f"Amount -- Fraud  : median=${fraud['Amount'].median():.2f}  "
      f"mean=${fraud['Amount'].mean():.2f}  max=${fraud['Amount'].max():.2f}")
print(f"Amount -- Legit  : median=${legit['Amount'].median():.2f}  "
      f"mean=${legit['Amount'].mean():.2f}  max=${legit['Amount'].max():.2f}")
print()
print(f"Time spans {df['Time'].max() / 3600:.1f} hours "
      f"(~{df['Time'].max() / 3600 / 24:.1f} days) of transactions.")


Amount -- Fraud  : median=$9.25  mean=$122.21  max=$2125.87
Amount -- Legit  : median=$22.00  mean=$88.29  max=$25691.16

Time spans 48.0 hours (~2.0 days) of transactions.


## 6 · Stratified train/test split

**Why stratified?**
With only 492 fraud cases, a random split risks placing almost all fraud in one
fold.  Stratified splitting guarantees the 0.17% fraud rate is preserved in
*both* halves -- so the test set actually reflects production conditions.

**Why 80/20?**
We need enough test-set fraud cases (≥ 90) to get reliable metric estimates
at the extremes of the PR curve.  80/20 on 284,807 rows gives us ~98 fraud
cases in test -- sufficient.

**Critical rule**: the test set is saved here and **never modified again**.
SMOTE, scaling, and feature engineering all happen inside the training fold only.


In [8]:
train, test = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["Class"],
)

print(f"Train : {len(train):>7,} rows  |  fraud: {train['Class'].sum():>4}  "
      f"({train['Class'].mean()*100:.4f}%)")
print(f"Test  : {len(test):>7,} rows  |  fraud: {test['Class'].sum():>4}  "
      f"({test['Class'].mean()*100:.4f}%)")
print()
print(f"Fraud rate preserved -- train {train['Class'].mean()*100:.4f}% "
      f"vs test {test['Class'].mean()*100:.4f}%  (should be ~equal)")


Train : 227,845 rows  |  fraud:  394  (0.1729%)
Test  :  56,962 rows  |  fraud:   98  (0.1720%)

Fraud rate preserved -- train 0.1729% vs test 0.1720%  (should be ~equal)


## 7 · Save splits

Both splits are saved as CSV to `data/`.  Every subsequent notebook loads
from these files rather than re-splitting, ensuring reproducibility.


In [9]:
train.to_csv(DATA_DIR / "train.csv", index=False)
test.to_csv(DATA_DIR / "test.csv",  index=False)

print(f"Saved data/train.csv  ({len(train):,} rows x {train.shape[1]} cols)")
print(f"Saved data/test.csv   ({len(test):,} rows x {test.shape[1]} cols)")


Saved data/train.csv  (227,845 rows x 31 cols)
Saved data/test.csv   (56,962 rows x 31 cols)


## Summary

| | Value |
|---|---|
| Total rows | 284,807 |
| Features | 30 (V1-V28, Time, Amount) + 1 target |
| Nulls | 0 |
| Fraud rate | 0.1727% |
| Train / Test | 227,845 / 56,962 |
| Train fraud | 394 |
| Test fraud | 98 |

The extreme 577:1 imbalance is the central challenge.
**Next notebook**: EDA visualisations + feature engineering + SMOTE resampling.
